# LIME on AfroXLMR (CPU — Google Drive)

**Before running:**
1. Upload `model_zul.zip`, `model_cross.zip`, and `zul_test.csv` to your Google Drive root
2. Set runtime to CPU (no GPU needed)
3. Run all cells (~30-60 min)

In [ ]:
!pip install transformers scikit-learn lime matplotlib pandas numpy scipy -q
print('Done')

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import zipfile, os, shutil

# Change these paths if your files are in a subfolder
DRIVE = '/content/drive/MyDrive'

for zname, dest in [('model_zul.zip', '/content/model_zul'), ('model_cross.zip', '/content/model_cross')]:
    src = f'{DRIVE}/{zname}'
    if os.path.exists(src):
        os.makedirs(dest, exist_ok=True)
        print(f'Extracting {zname}...')
        with zipfile.ZipFile(src, 'r') as zf:
            zf.extractall(dest)
        # Flatten if extracted into subfolder
        subdirs = [d for d in os.listdir(dest) if os.path.isdir(f'{dest}/{d}')]
        if len(subdirs) == 1 and 'config.json' not in os.listdir(dest):
            sub = f'{dest}/{subdirs[0]}'
            for f in os.listdir(sub):
                shutil.move(f'{sub}/{f}', f'{dest}/{f}')
            os.rmdir(sub)
        print(f'  -> {os.listdir(dest)}')
    else:
        print(f'NOT FOUND: {src}')

# Copy test CSV
test_src = f'{DRIVE}/zul_test.csv'
if os.path.exists(test_src):
    shutil.copy(test_src, '/content/zul_test.csv')
    print('Copied zul_test.csv')
else:
    print(f'NOT FOUND: {test_src}')

In [ ]:
import torch
import numpy as np
import pandas as pd
import scipy.special
from transformers import AutoTokenizer, AutoModelForSequenceClassification

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using: {DEVICE}')

tokenizer = AutoTokenizer.from_pretrained('/content/model_zul')
model_zul = AutoModelForSequenceClassification.from_pretrained('/content/model_zul').to(DEVICE).eval()
model_cross = AutoModelForSequenceClassification.from_pretrained('/content/model_cross').to(DEVICE).eval()
print('Models loaded')

In [ ]:
test_df = pd.read_csv('/content/zul_test.csv')
samples = pd.concat([
    test_df[test_df['label'] == 0].sample(5, random_state=42),
    test_df[test_df['label'] == 1].sample(5, random_state=42)
]).reset_index(drop=True)
print(f'LIME samples: {len(samples)} (5 human + 5 machine)')

In [ ]:
def make_predictor(model):
    def predict_proba(texts):
        all_probs = []
        for i in range(0, len(texts), 8):
            batch = texts[i:i+8]
            inputs = tokenizer(batch, padding=True, truncation=True, max_length=256, return_tensors='pt').to(DEVICE)
            with torch.no_grad():
                logits = model(**inputs).logits.cpu().numpy()
            all_probs.append(scipy.special.softmax(logits, axis=-1))
        return np.vstack(all_probs)
    return predict_proba

predict_zul = make_predictor(model_zul)
predict_cross = make_predictor(model_cross)
print('Predictors ready')

In [ ]:
from lime.lime_text import LimeTextExplainer
from collections import defaultdict
import time

explainer = LimeTextExplainer(class_names=['Human', 'Machine'])
os.makedirs('/content/lime_output', exist_ok=True)

def run_lime(predictor, model_name, samples_df):
    token_weights = defaultdict(list)
    start = time.time()
    for idx in range(len(samples_df)):
        row = samples_df.iloc[idx]
        print(f'  [{model_name}] Sample {idx+1}/{len(samples_df)}...')
        exp = explainer.explain_instance(
            row['text'], predictor,
            num_features=12, num_samples=300
        )
        exp.save_to_file(f'/content/lime_output/{model_name}_sample{idx}.html')
        for word, weight in exp.as_list():
            token_weights[word].append(weight)
    elapsed = time.time() - start
    print(f'  Done in {elapsed/60:.1f} min')
    rows = []
    for token, weights in token_weights.items():
        avg = np.mean(weights)
        rows.append({'experiment': model_name, 'model': 'afro-xlmr-base', 'token': token, 'avg_weight': round(abs(avg), 4), 'direction': 'Machine' if avg > 0 else 'Human'})
    return pd.DataFrame(rows).sort_values('avg_weight', ascending=False).head(12)

print('Running LIME — isiZulu model...')
lime_zul = run_lime(predict_zul, 'zul', samples)

print('\nRunning LIME — Cross-lingual model...')
lime_cross = run_lime(predict_cross, 'cross', samples)

In [ ]:
all_lime = pd.concat([lime_zul, lime_cross], ignore_index=True)
all_lime.to_csv('/content/lime_output/all_afroxlmr_lime_tokens.csv', index=False)

print('=== isiZulu Model Top Tokens ===')
print(lime_zul.to_string(index=False))
print('\n=== Cross-lingual Model Top Tokens ===')
print(lime_cross.to_string(index=False))

top_zul_set = set(lime_zul['token'].tolist())
top_cross_set = set(lime_cross['token'].tolist())
jaccard = len(top_zul_set & top_cross_set) / len(top_zul_set | top_cross_set)
print(f'\nJaccard similarity: {jaccard:.3f}')

In [ ]:
# Save to Drive and download
shutil.make_archive('/content/afroxlmr_lime_results', 'zip', '/content/lime_output')
shutil.copy('/content/afroxlmr_lime_results.zip', f'{DRIVE}/afroxlmr_lime_results.zip')
print('Saved to Google Drive: afroxlmr_lime_results.zip')

from google.colab import files
files.download('/content/afroxlmr_lime_results.zip')